# Example 1.2 / 1: Scalar and Indexed Parameters

Companion notebook for **1.2 Parameters** in the AMPL lab Documentation, section "Scalar and Indexed Parameters". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Scalar and Indexed Parameters

`param avail;` declares one number. Adding an indexing expression declares a whole collection:
`param rate {PROD};` has one value `rate[p]` for each product `p` in `PROD`. Think of an indexed
parameter as a Python dictionary whose keys are the set's members.

The production model from the AMPL book uses both kinds. The model declares the data, and
Python gives the values. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already
read `model.mod`. `ampl.param["rate"]` (or `ampl.get_parameter("rate")`) is the `Parameter`
object: assign a number to a scalar parameter, and a dictionary keyed by the set's members to an
indexed one. `value()` reads a scalar back, and `to_pandas()` returns an indexed parameter as a
pandas DataFrame:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                    # products
param rate {PROD} > 0;       # tons produced per hour
param profit {PROD};         # profit per ton
param market {PROD} >= 0;    # most that can be sold, in tons
param avail >= 0;            # hours available

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
print(ampl.param["avail"].value())
print(ampl.param["rate"].to_pandas())

40
       rate
bands   200
coils   140


### Pure AMPL: data.dat and model.run

Several parameters indexed over the same set can also be set at once from one DataFrame, with
one column per parameter: `ampl.set_data(df, "PROD")` sets the index as the set's members and
each column as the parameter of the same name.

The same in pure AMPL, with the same `model.mod`. In a data file, several parameters indexed over
the same set can share one table, and `display` prints them side by side:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param:  rate  profit  market :=
  bands  200    25     6000
  coils  140    30     4000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display avail;
display rate, profit, market;

avail = 40

:      rate profit market    :=
bands   200    25    6000
coils   140    30    4000
;

